<a href="https://colab.research.google.com/github/flexfengfeng/SMU-Hackathon-Prep/blob/main/workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Hackathon Masterclass: Build a Grounded AI Help Desk with RAG

Welcome to the interactive coding lab! In this notebook, we will build a Retrieval-Augmented Generation (RAG) system from scratch.

**What you will build:**
An AI help desk for **BridgeWorks SG–IN**, a fictional programme that helps small businesses in Singapore and India find partners in the other country. The bot answers *only* from the programme's own documents, shows **which document** each answer came from, and says **"I don't know"** when the documents don't cover the question.

**Why this matters for every track:** all four sponsors want answers people can trust. Judges will ask *"where did that come from?"* and *"what happens when the AI doesn't know?"*. This notebook teaches the building blocks your track notebook uses:

| Step | Technique | Used again in |
|---|---|---|
| 2 | Retry on rate limits | All tracks |
| 3–4 | Chunking, embeddings, vector search | Track 2 (cross-lingual search) |
| 5 | Grounded prompt with an "I don't know" rule | All tracks |
| 6 | Structured output with source IDs and a citation check | Tracks 1–4 |
| 7 | Gradio web UI | All tracks |

> ⚠️ BridgeWorks SG–IN, its fees and its contact details are **fictional**. Do not use them as real business or trade advice.

## Step 1: Install Dependencies
We need three main libraries:
- `google-genai`: Google's official Gen AI SDK, to access Gemini models for embeddings and text generation (already preinstalled in Colab).
- `chromadb`: A lightweight vector database to store our document chunks.
- `gradio`: To create a beautiful, interactive web interface for our bot.

`pydantic` is already in Colab.

The first line records the versions of a few core packages Colab depends on, so the install below doesn't upgrade them and break Colab. If Colab asks you to **restart the session** after installing, click restart and continue from Step 2.

In [ ]:
# Keep Colab's own versions of these packages to avoid dependency conflicts
!pip freeze | grep -iE '^(google-auth|opentelemetry-(api|sdk))==' > /tmp/colab-constraints.txt
!pip install -q -c /tmp/colab-constraints.txt google-genai chromadb==1.5.9 gradio==6.29.1

## Step 2: Setup Google Gemini API Key
1. Go to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your Google account and click **Get API key**.
3. In Colab, click the 🔑 **Secrets** icon on the left sidebar.
4. Create a new secret named `GEMINI_API_KEY` and paste your key.
5. Turn on the **Notebook access** toggle for that secret.

We also set the client to wait and retry when the free tier says *"429: too many requests"*. Free-tier keys only allow about 5 requests per minute, so every team will hit this during the hackathon. If a cell seems to pause for up to a minute, it is waiting out the limit, not stuck.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

# Free-tier keys only allow a few requests per minute. These options make the SDK
# wait and retry any single request that gets "429: too many requests".
RETRY = types.HttpRetryOptions(attempts=6, initial_delay=15, max_delay=60, http_status_codes=[429, 500, 503])

# Securely load the API key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY, http_options=types.HttpOptions(retry_options=RETRY))
    print("✅ Gemini API Key configured successfully!")
except Exception as e:
    print("❌ Error: Please ensure you added 'GEMINI_API_KEY' to your Colab Secrets and turned on 'Notebook access'.")
    print("   Details:", e)
    raise

# One place to change the model for the whole notebook
MODEL = "gemini-3.5-flash-lite"

def ask_gemini(contents, config=None):
    """Call Gemini. Rate-limit retries are handled by the client (see RETRY above)."""
    return client.models.generate_content(model=MODEL, contents=contents, config=config)

## Step 3: Create the Knowledge Base (Ingestion)
Our knowledge base is a small set of help pages for the fictional **BridgeWorks SG–IN** programme.

**Chunking:** we split the knowledge base into pieces small enough to search. Here each help page is already one topic, so **one page = one chunk**. Each chunk gets an **ID** (like `fees`) so the bot can later tell us exactly where an answer came from.

> 💡 **Hackathon tip:** for a long PDF, split by headings first, then by paragraphs. Keep the heading in each chunk so it still makes sense on its own.

In [ ]:
# A fictional programme. All names, fees and contacts are made up for this workshop.
DOCS = [
    {"id": "about", "title": "What BridgeWorks SG–IN does", "text":
     "BridgeWorks SG–IN helps small and medium businesses (fewer than 200 staff) in Singapore and India "
     "find customers, suppliers and partners in the other country. We introduce you to vetted partners, "
     "run market briefings and organise trade missions."},
    {"id": "fees", "title": "Membership plans and fees", "text":
     "Starter plan: free, includes 2 partner introductions per year. "
     "Growth plan: S$480 or ₹28,000 per year, includes 10 partner introductions, all briefing recordings "
     "and 30% off trade missions. You can get a full refund within 14 days of joining if you have not used "
     "any introductions."},
    {"id": "matchmaking", "title": "How partner matchmaking works", "text":
     "Fill in a one-page business profile. Within 10 working days we suggest 3 possible partners, and "
     "send an introduction email to each one you choose. We check that partners are registered businesses, "
     "but we do not guarantee that a deal will happen."},
    {"id": "briefings", "title": "Monthly market briefings", "text":
     "A free one-hour online briefing every month, alternating between the Singapore and Indian markets. "
     "Briefings are in English with Hindi and Chinese subtitles. Recordings are available to Growth members only."},
    {"id": "missions", "title": "Trade missions", "text":
     "We lead two trade missions a year: Singapore in March and Bengaluru in September. Each mission has "
     "3 days of business meetings and site visits. Register at least 6 weeks before departure. "
     "Travel and hotel costs are paid by each participant."},
    {"id": "limits", "title": "What we do not do", "text":
     "BridgeWorks does not give loans or grants, does not give legal or tax advice, and does not handle "
     "customs clearance or shipping. For these, we refer you to licensed professionals and the official "
     "government agencies in each country."},
    {"id": "contact", "title": "Contact and support hours", "text":
     "Email help@bridgeworks.example. Phone support is Monday to Friday, 9 AM to 6 PM Singapore time "
     "(6:30 AM to 3:30 PM India time). Our WhatsApp bot answers simple questions 24/7."},
]

chunks = [f"{d['title']}\n{d['text']}" for d in DOCS]   # keep the title in the chunk text
ids = [d["id"] for d in DOCS]
titles = {d["id"]: d["title"] for d in DOCS}

print(f"Created {len(chunks)} document chunks.")
print("Sample chunk:", ids[0], "->", chunks[0])

## Step 4: Embed & Store in Vector DB
To search by **meaning** instead of exact words, we turn each chunk into an **embedding**: a list of numbers where similar meanings end up close together. We store them in `ChromaDB`, a vector database.

Notice two things:
- The `task_type`: documents are embedded as `RETRIEVAL_DOCUMENT` and questions as `RETRIEVAL_QUERY`. Gemini optimises each side differently, which gives better search results.
- We embed **all chunks in one request**, not one request per chunk. With a 5-requests-per-minute limit, batching is the difference between waiting 0 seconds and waiting a minute.

In [ ]:
import chromadb

# Initialize ChromaDB in-memory
chroma_client = chromadb.Client()

# Create a collection to hold our documents.
# Delete any old copy first so this cell is safe to re-run.
COLLECTION_NAME = "bridgeworks_knowledge_base"
if COLLECTION_NAME in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(COLLECTION_NAME)
collection = chroma_client.create_collection(name=COLLECTION_NAME)

EMBED_MODEL = "gemini-embedding-001"

def get_embeddings(texts, task_type="RETRIEVAL_DOCUMENT"):
    """Embed a list of texts in ONE request."""
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return [e.values for e in result.embeddings]

# Embed every chunk in one call and add them to the database
collection.add(
    ids=ids,
    embeddings=get_embeddings(chunks),
    documents=chunks,
)

print(f"✅ {collection.count()} chunks embedded and stored in ChromaDB!")

## Step 5: The RAG Core (Retrieve + Generate)
When a user asks a question:
1. We embed their question.
2. We search ChromaDB for the closest chunks.
3. We pass the question AND those chunks to Gemini, with a rule: **answer only from the context, otherwise say you don't know**.

That last rule is what stops the bot from making things up (hallucinating).

In [ ]:
def retrieve(question, k=3):
    """Return the k chunks closest in meaning to the question, as (id, text) pairs."""
    query_embedding = get_embeddings([question], task_type="RETRIEVAL_QUERY")[0]
    results = collection.query(query_embeddings=[query_embedding], n_results=k)
    return list(zip(results["ids"][0], results["documents"][0]))

def build_prompt(question, retrieved):
    context = "\n\n".join(f"[{cid}] {text}" for cid, text in retrieved)
    return f"""You are a polite, helpful help-desk assistant for BridgeWorks SG–IN.
Use ONLY the information in the context below to answer the user's question.
If the answer is not in the context, say "Sorry, I don't have that information. Please contact our team."

Context:
{context}

User question: {question}"""

def respond(question):
    retrieved = retrieve(question)
    response = ask_gemini(build_prompt(question, retrieved))
    return response.text or "Sorry, I couldn't generate an answer. Please try rephrasing."

# Let's test it in the console! (3 requests: 2 embeddings + 1 generation)
test_question = "How much does the Growth plan cost in rupees?"
print("Question:", test_question)
print("Retrieved:", [cid for cid, _ in retrieve(test_question)])
print("Answer:\n", respond(test_question))

## Step 6: Make Answers Checkable (Structured Output + Citations)
A plain-text answer is hard to check. Judges, and the sponsors' compliance teams, will ask: **which document says that?**

So we ask Gemini to reply in a fixed shape, defined with a `pydantic` class:
- `answer`: the reply to show the user,
- `source_ids`: the IDs of the chunks it used,
- `answered_from_context`: `False` if the context did not contain the answer.

Gemini fills this in as JSON, and `response.parsed` turns it straight into a Python object.

Then **Python checks the AI's work**: any source ID that was not actually retrieved is a made-up citation, so we drop it. Never trust a citation you didn't verify.

We test two questions:
1. one the documents **do** answer, and
2. one they **don't** (an Indian tax question). A good bot says it doesn't know instead of guessing.

In [ ]:
from pydantic import BaseModel, Field

class GroundedAnswer(BaseModel):
    answer: str = Field(description="Short, friendly answer for the user")
    source_ids: list[str] = Field(description="IDs of the context chunks the answer is based on, e.g. ['fees']")
    answered_from_context: bool = Field(description="False if the context does not contain the answer")

ANSWER_CONFIG = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_schema=GroundedAnswer,
)

def grounded_answer(question):
    retrieved = retrieve(question)
    result = ask_gemini(build_prompt(question, retrieved), config=ANSWER_CONFIG).parsed
    if result is None:
        return GroundedAnswer(answer="Sorry, I couldn't generate an answer. Please try rephrasing.",
                              source_ids=[], answered_from_context=False)
    # Citation check: keep only IDs we actually retrieved
    retrieved_ids = {cid for cid, _ in retrieved}
    result.source_ids = [s for s in result.source_ids if s in retrieved_ids]
    if not result.source_ids:
        result.answered_from_context = False
    return result

# 4 requests: 2 questions x (1 embedding + 1 generation)
for q in ["Can BridgeWorks lend my company money to expand to India?",
          "What is the GST rate for software services in India?"]:
    a = grounded_answer(q)
    print("Q:", q)
    print("A:", a.answer)
    print("   sources:", a.source_ids or "none", "| answered from context:", a.answered_from_context, "\n")

## Step 7: Build the Web UI with Gradio 🏆 (Hackathon Winning Move)
Terminal output is boring. Judges love clean UIs. We will use Gradio to turn our Python function into a web application in seconds.

Every reply shows its **sources**, so anyone can check the answer against the original help page.

> **Note:** this bot answers each message on its own. It does not use the chat `history`, so follow-up questions like "and the free plan?" won't remember the previous message. Adding memory is a great stretch goal for your hackathon project!

In [ ]:
import gradio as gr

# Wrapper function for Gradio interface format
# (history is ignored: each question is answered independently)
def chat_interface(message, history):
    a = grounded_answer(message)
    if not a.answered_from_context:
        return a.answer + "\n\n_⚠️ Not found in our help pages._"
    sources = "\n".join(f"- `{s}`: {titles[s]}" for s in a.source_ids)
    return f"{a.answer}\n\n**Sources**\n{sources}"

# Create the Chat UI
ui = gr.ChatInterface(
    fn=chat_interface,
    title="🤝 BridgeWorks SG–IN Help Desk",
    description="Ask about plans, matchmaking, briefings or trade missions. Every answer shows its sources. Powered by Gemini & RAG.",
    examples=["Is there a free plan?", "When is the next trade mission to India?", "Can you help me clear customs?"],
)

# Launch it! (This will generate a public URL you can share with judges)
ui.launch(share=True, theme="soft")

## Step 8: From Here to Your Track 🧭
You now have the core pattern: **retrieve trusted information ➜ answer only from it ➜ show the source ➜ admit when you don't know.** Each track notebook builds on it:

| Track | Sponsor | What carries over |
|---|---|---|
| 1 · Emerging Technology | Singtel | Structured output with **citation checks**: every account brief cites the CRM records and buying signals behind it, and made-up IDs are removed. Adds **function calling** so the agent picks which data to look up. |
| 2 · Social Innovation | raiSE | **Embeddings** for cross-lingual search: ask in Chinese or Hindi, find English descriptions. Uses plain NumPy instead of ChromaDB, which is enough for a few dozen items. |
| 3 · Fintech / InsurTech | Income | Structured output for multilingual SMS, plus an **LLM-as-judge** that checks whether a customer could say back what the product does. |
| 4 · Sustainability | METI | **Vectorless RAG**: when a document has clear headings (like a quality standard), Gemini picks sections from the table of contents instead of using embeddings. |

**Discussion questions**
1. Ask the bot "Do you give tax advice?" and then "What's the tax rate in Singapore?". Why should the answers be different?
2. What would go wrong if we set `k=1` in `retrieve()`? What about `k=7` (every chunk)?
3. Your track's data is probably messier than seven short help pages. How would you chunk a 40-page PDF?